In [ ]:
# ============================================================
# 1. Python 3.11 環境の構築
#    (Colab デフォルトの 3.12 では DragGAN の依存が動かないため)
# ============================================================
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get -y update
!sudo apt-get -y install python3.11
!sudo apt-get -y install python3.11-dev
!sudo apt-get -y install python3-pip
!sudo apt-get -y install python3.11-distutils

!python3.11 -m pip install --upgrade setuptools
!python3.11 -m pip install --upgrade pip
!python3.11 -m pip install --upgrade distlib

!sudo update-alternatives --install /usr/bin/python python /usr/bin/python3.11 1
!sudo update-alternatives --set python /usr/bin/python3.11
!sudo ln -sf /usr/bin/python /usr/local/bin/python

!python -V

In [ ]:
# ============================================================
# 2. リポジトリの取得と依存インストール
#    requirements.txt は最新環境向けにバージョンを固定済み
# ============================================================
!git clone https://github.com/osdym0216/DragGAN_reproduction.git
%cd DragGAN_reproduction
!pip install -r requirements.txt

In [ ]:
# ============================================================
# 3. 学習済みモデル(StyleGAN2 重み)のダウンロード
# ============================================================
!python scripts/download_model.py

In [ ]:
# ============================================================
# 4. CUDA カスタムオペレータのビルド確認
# ============================================================
import os

%cd /content/DragGAN_reproduction
os.environ['CUDA_HOME'] = '/usr/local/cuda'

# 壊れた/中途半端なビルドキャッシュを掃除し、コンパイルの冗長性を有効にする
!rm -rf /root/.cache/torch_extensions ~/.cache/torch_extensions
import torch_utils.custom_ops as c
c.verbosity = 'full'

# パッチ済み custom_ops 経由で3つのプラグインをこのカーネルにロードする
from torch_utils.ops import bias_act, upfirdn2d, filtered_lrelu
print('bias_act     :', bias_act._init())        # True が出ればロード成功
print('upfirdn2d    :', upfirdn2d._init())
print('filtered_lrelu:', filtered_lrelu._init())

In [ ]:
# ============================================================
# 5. 【用途A】GUI の起動(インタラクティブに編集を試す)
# ============================================================
!MPLBACKEND=Agg python visualizer_drag_gradio.py

In [ ]:
# ============================================================
# 6. 【用途B】失敗分析・実験の実行
#    ランドマーク検出モデルを取得してから実験スクリプトを実行
# ============================================================
!wget -q -O experiments/face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!MPLBACKEND=Agg python -m experiments.lambda_wink_experiment